In [ ]:
# =============================================================================
# 第1章：预备知识 - 数据操作与可视化基础
# =============================================================================
# %matplotlib inline 是 Jupyter 的魔法命令，作用是让 matplotlib 绘制的图表
# 直接显示在 notebook 的 cell 输出中，而不是弹出单独的窗口
%matplotlib inline

# 导入 NumPy 库，用于高效的数值计算和数组操作
# NumPy 是 Python 科学计算的基础库，提供了多维数组对象和各种数学函数
import numpy as np

# 从 matplotlib_inline 导入 backend_inline，用于配置 Jupyter 的绘图后端
# 这个库专门用于优化 matplotlib 在 Jupyter 环境中的显示效果
from matplotlib_inline import backend_inline

# 导入 d2l 库中的 torch 模块，这是《动手学深度学习》配套的实用函数库
# d2l 提供了许多便利的绘图和训练辅助函数
from d2l import torch as d2l


# =============================================================================
# 导数计算函数：数值求导法
# =============================================================================
# 这个函数使用数值方法（差分法）计算函数 f 在点 x 处的导数近似值
# 数值求导的公式：f'(x) ≈ (f(x+h) - f(x)) / h，其中 h 是一个很小的数
def numerical_lim(f, x, h):
    """
    参数:
        f: 要求导的目标函数
        x: 求导点的位置
        h: 微小的增量（步长），越小精度越高但可能引入数值误差
    返回:
        导数的数值近似值
    """
    return (f(x + h) - f(x)) / h


# =============================================================================
# 定义示例函数：二次函数
# =============================================================================
# 定义一个简单的二次函数 f(x) = 3x² - 4x
# 这个函数的导数是 f'(x) = 6x - 4
# 在 x=1 处的导数值应该是 f'(1) = 6(1) - 4 = 2
def f(x):
    return 3 * x ** 2 - 4 * x


# =============================================================================
# 绘图配置函数
# =============================================================================
# 下面的函数封装了 matplotlib 的常用配置，用于统一 notebook 中的图表样式

def use_svg_display():  #@save
    """
    使用 SVG 格式在 Jupyter 中显示绘图
    SVG 是矢量图形格式，相比 PNG 位图有以下优势：
    1. 放大不会失真，清晰度高
    2. 文件体积通常更小
    3. 支持交互和动画
    """
    # 设置 matplotlib 输出格式为 SVG
    backend_inline.set_matplotlib_formats('svg')


def set_figsize(figsize=(3.5, 2.5)):  #@save
    """
    设置 matplotlib 的图表大小
    
    参数:
        figsize: 元组 (宽, 高)，单位是英寸
                 默认值 3.5x2.5 英寸适合论文插图
                 可以根据需要调整，比如 (7, 5) 适合演示
    """
    # 先调用 SVG 设置，确保输出格式正确
    use_svg_display()
    # 通过 rcParams 全局设置图表尺寸
    # rcParams 是 matplotlib 的运行时配置参数
    d2l.plt.rcParams['figure.figsize'] = figsize


#@save
def set_axes(axes, xlabel, ylabel, xlim, ylim, xscale, yscale, legend):
    """
    设置 matplotlib 的坐标轴属性
    
    这个函数统一配置图表的各个元素，保持样式一致性
    
    参数:
        axes: matplotlib 的坐标轴对象
        xlabel: x 轴标签文字
        ylabel: y 轴标签文字
        xlim: x 轴显示范围 (min, max)
        ylim: y 轴显示范围 (min, max)
        xscale: x 轴刻度类型 ('linear' 线性或 'log' 对数)
        yscale: y 轴刻度类型
        legend: 图例文字列表，None 表示不显示图例
    """
    # 设置 x 轴标签
    axes.set_xlabel(xlabel)
    # 设置 y 轴标签
    axes.set_ylabel(ylabel)
    # 设置 x 轴刻度类型（线性或对数）
    axes.set_xscale(xscale)
    # 设置 y 轴刻度类型
    axes.set_yscale(yscale)
    # 设置 x 轴显示范围
    axes.set_xlim(xlim)
    # 设置 y 轴显示范围
    axes.set_ylim(ylim)
    # 如果提供了图例文字，则显示图例
    if legend:
        axes.legend(legend)
    # 显示网格线，便于读数
    axes.grid()


#@save
def plot(X, Y=None, xlabel=None, ylabel=None, legend=None, xlim=None,
         ylim=None, xscale='linear', yscale='linear',
         fmts=('-', 'm--', 'g-.', 'r:'), figsize=(3.5, 2.5), axes=None):
    """
    绘制数据点的高级封装函数
    
    这是一个通用的绘图函数，支持：
    - 绘制单条或多条线
    - 自动处理输入数据的格式
    - 统一的样式配置
    
    参数:
        X: x 轴数据，可以是数组或数组的列表
        Y: y 轴数据，如果为 None，则 X 被视为 y 数据，x 使用索引
        xlabel, ylabel: 轴标签
        legend: 图例文字列表
        xlim, ylim: 坐标轴范围
        xscale, yscale: 坐标轴类型
        fmts: 线条格式列表，如 '-' 实线, '--' 虚线
        figsize: 图表尺寸
        axes: 指定的坐标轴对象，None 则使用当前坐标轴
    """
    # 如果 legend 为 None，初始化为空列表
    if legend is None:
        legend = []

    # 设置图表尺寸
    set_figsize(figsize)
    # 如果未提供 axes，获取当前坐标轴（如果当前没有则自动创建）
    axes = axes if axes else d2l.plt.gca()

    # 内部辅助函数：判断输入是否为单轴数据（一维数组或列表）
    def has_one_axis(X):
        """
        判断 X 是否只有一个轴（一维）
        返回 True 表示 X 是一维数组或普通列表（非嵌套）
        """
        return (hasattr(X, "ndim") and X.ndim == 1 or isinstance(X, list)
                and not hasattr(X[0], "__len__"))

    # 如果 X 是一维的，将其包装成列表，统一处理流程
    if has_one_axis(X):
        X = [X]
    
    # 如果 Y 为 None，说明只提供了 y 值，x 使用默认的索引
    if Y is None:
        # X 实际上是 y 数据，x 用空列表占位（后续会用索引）
        X, Y = [[]] * len(X), X
    # 如果 Y 是一维的，同样包装成列表
    elif has_one_axis(Y):
        Y = [Y]
    
    # 如果 X 和 Y 的长度不匹配，扩展 X 以匹配 Y 的数量
    # 这允许只提供一组 x 数据，多组 y 数据的情况
    if len(X) != len(Y):
        X = X * len(Y)
    
    # 清除当前坐标轴的所有内容，准备绘制
    axes.cla()
    
    # 遍历每组数据，分别绘制
    # zip(X, Y, fmts) 将 x 数据、y 数据、格式字符串一一对应
    for x, y, fmt in zip(X, Y, fmts):
        # 如果 x 不为空，使用给定的 x 和 y 绘制
        if len(x):
            axes.plot(x, y, fmt)
        # 如果 x 为空，y 使用索引作为 x（相当于 plot(y, fmt)）
        else:
            axes.plot(y, fmt)
    
    # 应用统一的坐标轴设置
    set_axes(axes, xlabel, ylabel, xlim, ylim, xscale, yscale, legend)


# =============================================================================
# 绘制函数图像和切线
# =============================================================================
# 创建 x 数据：从 0 到 3，步长 0.1
# np.arange 生成等差数列，类似于 Python 的 range，但支持浮点数
x = np.arange(0, 3, 0.1)

# 绘制函数 f(x) = 3x² - 4x 及其在 x=1 处的切线
# 切线方程：y = f'(1)(x-1) + f(1) = 2(x-1) + (-1) = 2x - 3
plot(x, [f(x), 2 * x - 3],           # x 数据和两组 y 数据
     'x',                            # x 轴标签
     'f(x)',                         # y 轴标签
     legend=['f(x)', 'Tangent line (x=1)'])  # 图例

In [ ]:
# =============================================================================
# 三维函数可视化
# =============================================================================
# 本节演示如何使用 Matplotlib 绘制三维曲面图
# 这在可视化损失函数、优化景观时非常有用

# 使用 Matplotlib 库绘制三维函数
import numpy as np  
import matplotlib.pyplot as plt  
# 从 mpl_toolkits.mplot3d 导入 Axes3D，这是 Matplotlib 的三维绘图工具
from mpl_toolkits.mplot3d import Axes3D  


# =============================================================================
# 定义三维函数
# =============================================================================
# 定义函数 f(x, y) = 3x³ + y⁴ + exp(y) + 5
# 这是一个非凸函数，包含多项式项和指数项
# 在机器学习中，类似的复杂曲面可能代表损失函数的形状
def f(x, y):  
    """
    计算三维函数的值
    
    参数:
        x: x 坐标值（可以是标量或数组）
        y: y 坐标值（可以是标量或数组）
    返回:
        函数 f(x,y) 的值
    """
    return 3 * x**3 + y**4 + np.exp(y) + 5  
  

# =============================================================================
# 创建网格数据
# =============================================================================
# np.linspace 生成等间距的数值序列
# 从 -2 到 2 生成 100 个点，包含端点
x = np.linspace(-2, 2, 100)  
y = np.linspace(-2, 2, 100)  

# np.meshgrid 将一维坐标数组转换为二维网格坐标矩阵
# 这是三维绘图的必备步骤
# 输入: x = [x1, x2, ...], y = [y1, y2, ...]
# 输出: X 是每行都是 x 的矩阵, Y 是每列都是 y 的矩阵
# 这样 f(X, Y) 可以计算整个网格上的函数值
X, Y = np.meshgrid(x, y)  

# 计算整个网格的 Z 值（函数值）
# 利用 NumPy 的广播机制，可以一次性计算整个网格
Z = f(X, Y)  


# =============================================================================
# 创建三维图形
# =============================================================================
# 创建一个新的图形对象
fig = plt.figure()  

# 添加三维子图
# 参数 111 表示：1行1列的第1个子图（即整个画布）
# projection='3d' 指定这是三维坐标轴
ax = fig.add_subplot(111, projection='3d')  

# 绘制三维曲面图
# plot_surface 绘制曲面，参数说明：
# - X, Y, Z: 网格坐标和高度数据
# - cmap='viridis': 颜色映射，viridis 是一种对色盲友好的渐变配色
#   其他常用配色：'plasma', 'inferno', 'magma', 'jet', 'coolwarm'
surf = ax.plot_surface(X, Y, Z, cmap='viridis')  

# 添加颜色条（colorbar）
# 颜色条显示高度值（Z值）与颜色的对应关系
fig.colorbar(surf)  

# =============================================================================
# 设置坐标轴标签
# =============================================================================
# 为三个坐标轴分别设置标签
ax.set_xlabel('X')  
ax.set_ylabel('Y')  
ax.set_zlabel('f(X, Y)')  

# 显示图形
plt.show()

In [ ]:
# =============================================================================
# 带噪声的函数采样与可视化
# =============================================================================
# 本节演示如何模拟真实的机器学习场景：
# 真实数据通常带有噪声，模型需要从噪声中学习规律
# 这里模拟在 f(x,y) 上添加高斯噪声的过程

import numpy as np  
import matplotlib.pyplot as plt  


# =============================================================================
# 定义带噪声的函数
# =============================================================================
# 在原有函数基础上添加随机噪声，模拟真实世界的测量误差
def f_with_noise(x, y):  
    """
    计算带噪声的函数值
    
    在机器学习场景中：
    - 真实函数 f(x,y) 代表数据生成的内在规律
    - 噪声 ε 代表测量误差、随机因素
    - 模型的目标是从 (x,y,z) 样本中还原 f(x,y)
    
    参数:
        x, y: 输入坐标
    返回:
        f(x,y) + ε，其中 ε ~ N(0, 0.01)
    """
    # np.random.normal 生成正态分布随机数
    # 参数：loc=0（均值），scale=0.1（标准差）
    # 注意：方差是 0.01，标准差是 sqrt(0.01) = 0.1
    epsilon = np.random.normal(0, 0.1)
    
    # 原函数：3x³ + y⁴ + exp(y) + 5
    return 3 * x**3 + y**4 + np.exp(y) + 5 + epsilon  


# =============================================================================
# 生成随机采样点
# =============================================================================
num_points = 1000  # 定义要生成的点的数量

# np.random.uniform 生成均匀分布的随机数
# 参数：low=-2（下限），high=2（上限），size=num_points（数量）
# 这模拟了在 [-2, 2] × [-2, 2] 区域内均匀随机采样
x_values = np.random.uniform(-2, 2, num_points)  
y_values = np.random.uniform(-2, 2, num_points)  

# 计算每个采样点的函数值（带噪声）
# 使用列表推导式遍历所有 (x, y) 对
z_values = [f_with_noise(x, y) for x, y in zip(x_values, y_values)]  


# =============================================================================
# 可视化采样结果
# =============================================================================
# 创建散点图，颜色表示 z 值的大小
# scatter 参数说明：
# - x_values, y_values: 点的坐标
# - c=z_values: 颜色映射的数据（值越大颜色越暖）
# - cmap='viridis': 颜色映射方案
# - alpha=0.6: 透明度（0完全透明，1完全不透明）
#              半透明可以让重叠的点更明显
plt.scatter(x_values, y_values, c=z_values, cmap='viridis', alpha=0.6)  

# 添加颜色条，显示颜色与 z 值的对应关系
plt.colorbar(label='f(x, y) + ε')  

# 设置坐标轴标签
plt.xlabel('x')  
plt.ylabel('y')  

# 设置图表标题，说明数据来源
plt.title('Scatter plot of f(x, y) + ε with x ~ U(-2, 2) and y ~ U(-2, 2)')  

# 显示图形
plt.show()

In [ ]:
# =============================================================================
# PyTorch 自动求导机制详解
# =============================================================================
# 自动求导（Autograd）是 PyTorch 的核心特性之一
# 它允许我们自动计算复杂函数的梯度，是神经网络训练的基础

import torch  


# =============================================================================
# 创建可求导的张量
# =============================================================================
# requires_grad=True 告诉 PyTorch 需要追踪这个张量的所有操作
# 这是构建计算图的关键，只有设置了这个标志的张量才会被记录梯度

# 创建张量 a，初始值为 2.0，需要计算梯度
# 可以把它想象成计算图的叶节点（输入节点）
a = torch.tensor(2.0, requires_grad=True)  

# 创建张量 b，初始值为 3.0，同样需要计算梯度
b = torch.tensor(3.0, requires_grad=True)  


# =============================================================================
# 构建计算图
# =============================================================================
# PyTorch 使用动态计算图（Dynamic Computation Graph）
# 每次运算都会自动构建计算图，运算结束后图会被释放

# 执行加法操作：c = a + b
# 这会创建一个新的计算节点 c，它依赖于 a 和 b
c = a + b  

# 执行乘法操作：d = a * b
# 这创建另一个计算节点 d，同样依赖于 a 和 b
d = a * b  

# 打印计算结果
print(f"a + b = {c}")  # 输出：5.0
print(f"a * b = {d}")  # 输出：6.0


# =============================================================================
# 反向传播计算梯度
# =============================================================================
# 在这个阶段，计算图已经隐式构建，但还没有计算梯度
# 需要调用 backward() 来触发反向传播

# 选择一个标量作为起点进行反向传播
# 在深度学习中，这通常是损失函数的值
# 这里我们简单地选择 d (a*b = 6.0) 作为示例
scalar_function = d  

# 调用 backward() 进行反向传播
# 它会：
# 1. 计算 scalar_function 对其直接输入的梯度
# 2. 使用链式法则，沿着计算图向后传播
# 3. 将所有叶节点的梯度累加到 .grad 属性
scalar_function.backward()  


# =============================================================================
# 查看计算结果
# =============================================================================
# a.grad 存储了标量函数对 a 的偏导数
# d = a * b，对 a 求偏导得到 b = 3.0
print(f"Gradient of a: {a.grad}")  # 输出：tensor(3.)

# b.grad 存储了标量函数对 b 的偏导数
# d = a * b，对 b 求偏导得到 a = 2.0
print(f"Gradient of b: {b.grad}")  # 输出：tensor(2.)


# =============================================================================
# 补充说明
# =============================================================================
# 如果 scalar_function = c + d = (a+b) + (a*b)
# 那么梯度应该是：
# ∂(c+d)/∂a = ∂c/∂a + ∂d/∂a = 1 + b = 4
# ∂(c+d)/∂b = ∂c/∂b + ∂d/∂b = 1 + a = 3
# 可以尝试修改代码验证这个结果！

In [ ]:
# =============================================================================
# 向量化计算与性能对比
# =============================================================================
# 本节通过对比 Python 循环和向量化的性能，展示 GPU/向量化计算的重要性
# 这是深度学习中必须掌握的核心概念：避免 Python 级别的循环

%matplotlib inline
import math          # 数学函数库
import time          # 时间测量
import numpy as np   # NumPy 数值计算
import torch         # PyTorch 深度学习框架
from d2l import torch as d2l  # d2l 工具函数


# =============================================================================
# 准备测试数据
# =============================================================================
# 定义向量长度，越大性能差异越明显
n = 10000

# 创建两个长度为 n 的全 1 向量
# torch.ones 创建全 1 张量，默认数据类型是 float32
a = torch.ones([n])
b = torch.ones([n])


# =============================================================================
# Timer 类：性能测量工具
# =============================================================================
class Timer:  #@save
    """
    记录多次运行时间的计时器类
    
    使用场景：
    - 测量训练 epoch 时间
    - 比较不同算法的性能
    - 分析代码瓶颈
    """
    
    def __init__(self):
        """初始化计时器，自动开始第一次计时"""
        self.times = []  # 存储每次 stop() 记录的时间
        self.start()     # 启动计时器

    def start(self):
        """
        启动/重置计时器
        记录当前时间作为起始点
        """
        # time.time() 返回当前时间戳（秒）
        self.tik = time.time()

    def stop(self):
        """
        停止计时器并将时间记录在列表中
        
        返回:
            本次计时的时间间隔（秒）
        """
        # 计算时间差 = 当前时间 - 起始时间
        elapsed = time.time() - self.tik
        self.times.append(elapsed)
        return elapsed

    def avg(self):
        """返回平均时间（所有记录时间的平均值）"""
        return sum(self.times) / len(self.times)

    def sum(self):
        """返回时间总和"""
        return sum(self.times)

    def cumsum(self):
        """
        返回累计时间数组
        用于绘制训练曲线时显示累计用时
        """
        # np.array(self.times).cumsum() 计算累计和
        # [1, 2, 3] → [1, 3, 6]
        return np.array(self.times).cumsum().tolist()


# =============================================================================
# 方法1：Python 循环（慢）
# =============================================================================
# 初始化结果向量
c = torch.zeros(n)

# 创建计时器
timer = Timer()

# 使用 Python for 循环逐元素相加
# 这是深度学习中应该避免的模式！
# 每次迭代都是 Python 级别的操作，效率极低
for i in range(n):
    c[i] = a[i] + b[i]

# 记录并打印时间（约 0.8 秒，取决于硬件）
print(f"Python 循环耗时: {timer.stop():.5f} sec")


# =============================================================================
# 方法2：向量化计算（快）
# =============================================================================
# 重置计时器
timer.start()

# 使用 PyTorch 的向量化操作
# 这是在底层 C++/CUDA 级别执行的，充分利用并行计算
d = a + b

# 记录并打印时间（约 0.001 秒，比循环快数百倍）
print(f"向量化计算耗时: {timer.stop():.30f} sec")


# =============================================================================
# 性能对比结论
# =============================================================================
# 向量化计算的优势：
# 1. 并行计算：CPU/GPU 可以同时处理多个元素
# 2. 内存局部性：数据连续存储，缓存命中率高
# 3. 避免 Python 开销：在底层 C++/CUDA 执行
#
# 实际建议：
# - 能用矩阵运算就不要用循环
# - 能用广播就不要用 expand
# - PyTorch/NumPy 提供的函数通常都经过优化

In [ ]:
# =============================================================================
# 正态分布（高斯分布）可视化
# =============================================================================
# 正态分布是深度学习中最基础的分布：
# - 权重初始化通常使用正态分布
# - 噪声模型常用高斯噪声
# - 中心极限定理保证了许多场景的正态性

%matplotlib inline
import torch
import math           # 提供 sqrt, pi, exp 等数学函数
from d2l import torch as d2l


# =============================================================================
# 正态分布的概率密度函数（PDF）
# =============================================================================
def normal(x, mu, sigma):
    """
    计算正态分布的概率密度函数值
    
    正态分布 PDF 公式：
    p(x) = (1 / √(2πσ²)) * exp(-(x-μ)² / (2σ²))
    
    其中：
    - μ (mu): 均值，分布的中心位置
    - σ (sigma): 标准差，控制分布的宽度
    - σ²: 方差
    
    参数:
        x: 要计算概率密度的点
        mu: 均值
        sigma: 标准差（必须 > 0）
    返回:
        p(x): 概率密度值
    """
    # 计算归一化系数：1 / √(2πσ²)
    # 这个系数保证整个概率密度函数的积分为 1
    p = 1 / math.sqrt(2 * math.pi * sigma**2)
    
    # 计算指数部分：exp(-(x-μ)² / (2σ²))
    # 这部分决定分布的形状，在 x=μ 处取得最大值 1
    return p * np.exp(-0.5 / sigma**2 * (x - mu)**2)


# =============================================================================
# 准备绘图数据
# =============================================================================
# 创建 x 轴数据：从 -7 到 7，步长 0.01
# 这个范围足够展示正态分布的主要特征
x = np.arange(-7, 7, 0.01)


# =============================================================================
# 定义不同的均值和标准差组合
# =============================================================================
# 我们绘制三条曲线来对比不同参数的影响：
# 1. (μ=0, σ=1): 标准正态分布
# 2. (μ=0, σ=2): 均值相同但方差更大（更平坦）
# 3. (μ=3, σ=1): 方差相同但均值偏移

params = [
    (0, 1),   # 标准正态分布，钟形曲线
    (0, 2),   # 标准差增大，曲线更扁平
    (3, 1)    # 均值偏移到 3，曲线整体右移
]


# =============================================================================
# 绘制正态分布曲线
# =============================================================================
# 使用列表推导式为每组参数计算对应的概率密度值
# [normal(x, mu, sigma) for mu, sigma in params] 生成三条曲线的 y 值

d2l.plot(
    x,                                          # x 轴数据
    [normal(x, mu, sigma) for mu, sigma in params],  # y 轴数据列表
    xlabel='x',                                 # x 轴标签
    ylabel='p(x)',                              # y 轴标签
    figsize=(4.5, 2.5),                         # 图表尺寸
    legend=[f'mean {mu}, std {sigma}' for mu, sigma in params]  # 图例
)


# =============================================================================
# 观察结论
# =============================================================================
# 1. μ（均值）控制曲线的中心位置
# 2. σ（标准差）控制曲线的宽度：
#    - σ 越大，曲线越扁平，数据越分散
#    - σ 越小，曲线越尖锐，数据越集中
# 3. 曲线下的总面积始终为 1（概率公理）